# Stage 2: LSQ — 让 scale 活起来

> ⏱ 预计学习时间：15-20 小时 | 🎯 难度：⭐⭐⭐
>
> Stage 1.5 的结尾你发现了固定 scale QAT 的硬伤：在 ≤4-bit 时，靠几个 epoch 的 Observer 决定 scale
> 然后固定它——精度直接崩。**LSQ 的出发点就是这个发现：如果 scale 能和 weight 一起被梯度优化，
> 它就能随着 weight 的变化自适应调整，永远保持在"当前 weight 分布下的最优值"。**
>
> 这个 Stage 的目标：从 STE 深潜开始，到手推 LSQ 的梯度公式，到从零实现，到 CIFAR-10 上的消融实验验证。

---

## 目录

1. [开篇：从 Stage 1.5 的"崩溃点"说起](#开篇)
2. [知识总览](#知识总览)
3. [1. STE 深潜：梯度如何穿过离散化](#1-ste-深潜)
4. [2. LSQ 梯度推导：一步一步手推公式 (6)](#2-lsq-梯度推导)
5. [3. Gradient Scaling：公式 (13) 的背后](#3-gradient-scaling)
6. [4. 从零实现 LSQ：完整的 autograd.Function](#4-从零实现-lsq)
7. [5. 和 PyTorch FakeQuantize 的对比：改哪一行就够了](#5-和-pytorch-fakequantize-的对比)
8. [6. LSQ+：zero_point 也可学习](#6-lsq)
9. [7. 动手实验](#7-动手实验)
10. [检验标准](#检验标准)

---

## 开篇：从 Stage 1.5 的"崩溃点"说起

Stage 1.5 的比特宽度实验给你看了一组数据：8-bit QAT 精度 91.0%（接近 FP32），4-bit 降到 87.8%，2-bit 崩到 68.5%。为什么会崩？

在 2-bit 下只有 4 个量化等级。如果 scale 偏了 20%，两个相邻等级之间就可能跳过最优权重——量化网格"对不准"了。而固定 scale 只靠前 2-3 个 epoch 的 Observer 决定——在训练初期权重还在快速变化时做出的决定，到训练后期可能完全不对。

LSQ 的解决方案很简单：**把 scale 变成 `nn.Parameter`，让它在 QAT 训练中和 weight 一起被梯度更新。** 但实现起来需要解决两个问题：

1. **scale 的梯度怎么算？** `round()` 切断了梯度链——你需要手动推导 `∂v̂/∂s`。
2. **scale 梯度的量级对吗？** 一个 scale 影响几百几千个权重——梯度天然比 weight 大得多，需要 Gradient Scaling。

---

## 知识总览

```
Stage 1.5 的发现: 固定 scale QAT 在低比特崩溃 → scale 的精确性至关重要

LSQ 的解决方案:
  scale = nn.Parameter(init_value)
  forward: v̂ = round(clamp(v/s, Qmin, Qmax)) × s  ← 和 FakeQuantize 一模一样
  backward: ∂L/∂s = 手动计算的梯度（公式 6 + 13）  ← 这是 LSQ 的核心

两条路线:
  LSQ:  对称量化, 只优化 scale（zero_point = 0）
  LSQ+: 非对称量化, scale 和 zero_point 都可学习
```

---

## 1. STE 深潜：梯度如何穿过离散化

### 1.1 `round()` 的导数 = 0（几乎处处）


### 回顾：Stage 1.5 给了我们什么

在进入 LSQ 的数学之前，先明确我们要解决什么问题。

Stage 1.5 的比特宽度实验给你展示了一组数据：
- 8-bit QAT: 精度接近 FP32（256 个等级，scale 不太精确也能凑合）
- 4-bit QAT: 开始下降（16 个等级，scale 偏差开始显现）
- 2-bit QAT: 直接崩溃（4 个等级，scale 必须极度精确）

**崩溃的根因不是"权重量化太粗"——是 scale 不够精确。**

在 2-bit 下只有 4 个量化等级。如果 scale 偏了 20%，
量化网格就会完全"对不准"权重的最优值。
而固定 scale 只靠前 2-3 个 epoch 的 Observer 决定——
在训练初期权重还在快速变化时做出的决定，到训练后期可能是错的。

**LSQ 的回答：让 scale 变成可学习参数，跟着 weight 一起被梯度优化。**

下面我们一步步实现这个想法。


In [ ]:
y = round(x)  # 前向: 离散化
∂y/∂x = 0     # 反向: 梯度断掉（除 x.5 处不可导）


### 1.4 从 STE 到 LSQ：就差一步

STE 解决了一个问题：如何让梯度穿过 `round()`。
但它留下了一个盲区：**scale 参数没有梯度。**

在普通 QAT 中这不是问题——因为 scale 是 buffer，本来就不参与训练。
但如果我们要让 scale 可学习——就需要手动给它设计梯度。

**LSQ 的公式 (6) 就是在回答：如果 scale 是可学习参数，它的梯度应该等于多少？**

下面的推导从乘法法则开始，一步步走到最终形式。
推导本身不难（只需要基础微积分+STE 的假设），
但每一步的**物理含义**才是重点——不要只盯着公式看，读每步后面的直觉解释。


如果用普通链式法则，梯度到 `round()` 就断了——模型收不到任何信号。

### 1.2 STE：假装 round() 不存在

```
前向: y = round(x)   ← 做真实的舍入
反向: ∂y/∂x = 1      ← 梯度直接穿过，不作修改
```

PyTorch 对 `torch.round()` 内建了 STE——你不需要手动写 autograd.Function，backward 时梯度自动直通。但 **scale 没有梯度**——标准 STE 只处理了输入 x，没有处理 scale 参数 s。

### 1.3 STE 的局限

低比特下，`round()` 的跳跃很大（2-bit 只有 4 个值，相邻差 scale 整个量级）。STE 说"梯度和原来一样"，但细微的 weight 变化可能让值从 level 0 跳到 level 2（完全跳过 level 1）——**STE 看不到这种跳跃。**

所以在低比特下，仅靠 STE 调整 weight 不够——你需要同时调整 scale，让量化网格跟着数据走。这就是 LSQ 要做的。

---

## 2. LSQ 梯度推导：一步一步手推公式 (6)

### 2.1 设三个中间变量

```
v_n = v / s                          — 缩放后的值
v̄ = clamp(v_n, Q_min, Q_max)         — clamp 后的值
v̂ = round(v̄) × s                     — 量化输出
```

### 2.2 乘法法则

```
∂v̂/∂s = ∂(round(v̄) × s) / ∂s
       = round(v̄) × ∂s/∂s + s × ∂round(v̄)/∂s
       = round(v̄) + s × ∂round(v̄)/∂s              (1)
```

### 2.3 链式展开 + STE

```
∂round(v̄)/∂s = ∂round(v̄)/∂v̄ × ∂v̄/∂s
STE: ∂round(v̄)/∂v̄ = 1
→ ∂round(v̄)/∂s = ∂v̄/∂s                            (2)
```

### 2.4 分三种情况

```
情况 A: Q_min < v/s < Q_max（在量化范围内）
  v̄ = v/s → ∂v̄/∂s = -v/s²

情况 B: v/s ≤ Q_min（被 clip 到下界）
  v̄ = Q_min → ∂v̄/∂s = 0

情况 C: v/s ≥ Q_max（被 clip 到上界）
  v̄ = Q_max → ∂v̄/∂s = 0
```

### 2.5 代回去

```
情况 A: ∂v̂/∂s = round(v̄) + s × (-v/s²) = round(v̄) - v/s
              = -v/s + round(v̄)                          ← ★ LSQ 公式 (6) !

情况 B: ∂v̂/∂s = round(Q_min) = Q_min

情况 C: ∂v̂/∂s = round(Q_max) = Q_max
```

### 2.6 直觉解读

在量化范围内：**`∂v̂/∂s = -v_n + round(v̄)`**——连续值在量化网格中的位置和离散网格点之间的差距。

- 差距 ≈ 0 → scale 很好，不调
- 差距为负 → scale 应该减小（让网格更密）
- 差距为正 → scale 应该增大（让网格更宽）

在 clip 外：梯度是 `Q_min` 或 `Q_max`——**自动引导 scale 减少 clip**。

LSQ 不需要额外的 loss 项。整个优化机制被封装在 backward 的梯度计算中，SGD 自动驱动。

---

## 3. Gradient Scaling：公式 (13) 的背后

### 3.1 问题

一个 scale 影响几百个 weight 的量化。所有 weight 的梯度汇到 1 个 scale 上——scale 梯度天然比 weight 梯度大几十到几百倍。不加控制，scale 会到处乱跑。

### 3.2 解决方案


In [ ]:
g = 1.0 / (N_W × Q_P) ** 0.5
grad_scale = grad_scale_raw × g


### 3.3 从梯度推导到代码：把公式变成 autograd.Function

公式 (6) 和 (13) 推导完后，下一步是把它们变成可运行的代码。

PyTorch 提供 `torch.autograd.Function` 让你自定义前向和反向逻辑。
LSQ 的 forward 和普通 FakeQuantize 完全一样——量化-反量化。
不同的只有 backward：手动计算 `grad_scale`（公式 6）+ Gradient Scaling（公式 13）。

**注意**：对输入 x 的梯度还是用 STE（和普通 FakeQuantize 一样）。
LSQ 只改变了 scale 的梯度——其他一切不变。


- `1/√(N_W)`：权重越多，每个 scale 承载的影响越大 → 更谨慎调整
- `1/√(Q_P)`：量化范围越大（8-bit vs 4-bit），scale 变化影响越大

### 3.3 具体算例

Conv2d(64, 128, 3×3), per-channel scale, 8-bit:
```
N_W = 128 × 64 × 9 = 73728,  Q_P = 127
g = 1 / √(73728 × 127) ≈ 1/3060 ≈ 0.00033
```
不用这个 g，scale 会振荡发散。

---

## 4. 从零实现 LSQ：完整的 autograd.Function


In [ ]:
class LSQFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, scale, n_bits, symmetric):
        qmax = 2**(n_bits-1)-1; qmin = -qmax-1 if symmetric else 0
        x_scaled = x / scale
        x_clamped = torch.clamp(x_scaled, qmin, qmax)
        x_rounded = torch.round(x_clamped)
        x_quant = x_rounded * scale
        ctx.save_for_backward(x, scale, x_scaled, x_clamped, x_rounded)
        ctx.qmin, ctx.qmax = qmin, qmax
        return x_quant

    @staticmethod
    def backward(ctx, grad_output):
        x, scale, x_scaled, x_clamped, x_rounded = ctx.saved_tensors
        qmin, qmax = ctx.qmin, ctx.qmax

        # 对 x: STE
        in_range = (x_scaled >= qmin) & (x_scaled <= qmax)
        grad_x = torch.where(in_range, grad_output, torch.zeros_like(grad_output))

        # 对 scale: LSQ 公式 (6)
        grad_s = -x_scaled + x_rounded
        grad_s = torch.where(in_range, grad_s,
                   torch.where(x_scaled < qmin, float(qmin), float(qmax)))
        grad_scale = (grad_output * grad_s).sum()

        # Gradient Scaling: 公式 (13)
        g = 1.0 / (x.numel() * max(abs(qmin), abs(qmax))) ** 0.5
        grad_scale *= g

        return grad_x, grad_scale, None, None


class LSQQuantizer(nn.Module):
    def __init__(self, n_bits=4, symmetric=True):
        super().__init__()
        self.n_bits, self.symmetric = n_bits, symmetric
        self.scale = nn.Parameter(torch.tensor(1.0))  # ← 关键: scale 是可学习的!

    def init_from_data(self, x):
        with torch.no_grad():
            self.scale.data = x.detach().abs().max() / (2**(self.n_bits-1)-1) * 0.8

    def forward(self, x):
        return LSQFunction.apply(x, self.scale, self.n_bits, self.symmetric)


### 7.3 LSQ 之后——回头看 Stage 1.5 的数据

学完 LSQ 的实现后，回顾 Stage 1.5 的比特宽度实验。

你会意识到：**LSQ 不是"一个更复杂的 QAT 方法"——它是"固定 scale QAT 在低比特下的必然出路"。**

固定 scale QAT 的能力边界在 4-bit 附近。LSQ 把这个边界推进到了 2-bit。
这不是"优化"——是"范式转换"：从"靠 Observer 猜一个 scale"到"让梯度自己找到最优 scale"。

这个范式转换的思想蔓延到了整个量化领域：
- AIMET 的 Range Learning QAT = LSQ 的工程化实现
- QLoRA 的 NF4 = LSQ 思想在非均匀量化上的应用
- BitNet 的三值化 = LSQ 在 1.58-bit 的极限探索

**你刚学完的不是一个"技巧"——是 QAT 领域最重要的理论基石。**


---

## 5. 和 PyTorch FakeQuantize 的对比：改哪一行就够了

```
PyTorch FakeQuantize:
  scale 是 buffer（register_buffer）→ optimizer 不更新
  backward 中 scale 的梯度 = None

LSQ:
  scale 是 nn.Parameter → optimizer 更新
  backward 中 scale 的梯度 = LSQ 公式 (6) + Gradient Scaling (13)

需要改的只有:
  1. register_buffer("scale") → nn.Parameter(scale_init)
  2. backward 中 grad_scale = None → grad_scale = LSQ 公式
  3. 前向完全不变。
```

---

## 6. LSQ+：zero_point 也可学习

LSQ 只优化 scale（对称量化，zero_point = 0）。高通的 LSQ+ 将其推广到非对称量化——zero_point 也可学习。

当激活值分布偏斜时（ReLU 后全 ≥ 0），非对称量化能更好利用量化范围。LSQ+ 让 zero_point 在 QAT 中也被梯度优化——本质是对 LSQ 的自然扩展。

---

## 7. 动手实验

| # | 实验 | 时间 |
|---|------|:--:|
| 1 | 在 MNIST 上用自实现 LSQ 跑 2/3/4/8-bit, 和固定 scale QAT 对比 | 1h |
| 2 | 画 LSQ scale 的学习曲线: 不同层的 scale 如何随时间变化 | 30min |
| 3 | 验证 Gradient Scaling: 不用 g 时 scale 的振荡图 | 20min |

**期望结果**（CIFAR-10, ResNet-20）:

```
┌──────────┬────────┬────────┬────────┬────────┐
│ Method   │ 8-bit  │ 4-bit  │ 3-bit  │ 2-bit  │
├──────────┼────────┼────────┼────────┼────────┤
│ Fixed QAT│ 91.0%  │ 87.8%  │ 81.2%  │ 68.5%  │
│ LSQ      │ 91.1%  │ 90.2%  │ 87.5%  │ 82.8%  │
└──────────┴────────┴────────┴────────┴────────┘
```

LSQ 在 2-bit 比固定 QAT 好 14 个点——这就是"让 scale 活起来"的价值。

---

## 检验标准

- [ ] 能手推 LSQ 公式 (6): `∂v̂/∂s = -v/s + round(v̄)`
- [ ] 能解释为什么 clip 外的梯度是 `Q_min` 或 `Q_max`
- [ ] 能从零实现 LSQ 的 `autograd.Function`（含 Gradient Scaling）
- [ ] 能在 CIFAR-10 上跑 LSQ vs Fixed QAT 的 8/4/3/2-bit 消融
- [ ] 能指出 PyTorch FakeQuantize backward 中"哪一行要改"来支持 LSQ
- [ ] 能画 LSQ scale 的学习曲线

---

> 💡 **学习建议**：LSQ 是整个学习路径的分水岭。攻克它的标志不是"看懂了公式"——是"能从白纸开始写出完整实现+跑通消融实验"。这时你回头再看 AIMET 的 Range Learning QAT、QLoRA 的 NF4——它们都是 LSQ 的变体。
>
> Next: [Stage 3: PTQ 进阶算法](./Stage3_PTQ进阶算法.md)
